# Self-play checkpoint evaluation

Evaluate checkpoints on one fixed set of 1,000 states after six turn-based throws. The first 500 states use the model on turns 7 and 9; the other 500 use it on turn 8.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / 'data_generation.py').exists():
    if repo_root.parent == repo_root:
        raise RuntimeError('Could not find repository root')
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import evaluation
from scratch import self_play_training as self_play

In [ ]:
# Evaluation parameters.
seed = 2026
num_states = 1000
checkpoint_indices = (1, 2, 5, 10, 20, 30)
grid_size = (10, 10, 4)
state_generator_version = 'turn_based'
max_stones = 10

artifact_dir = Path('/home/vietafan/curling_data/self_play_N5_round9_20k')
evaluation_dir = artifact_dir / 'evaluation'
evaluation_dir.mkdir(parents=True, exist_ok=True)
states_path = evaluation_dir / 'states_after_6_throws.npz'
if states_path.exists():
    states = evaluation.load_sheet_states(states_path)
else:
    states = self_play.seq.data_generation.generate_random_sheet_state_for_turn(
        turn=6, num_sims=num_states, rng=__import__('numpy').random.default_rng(seed)
    )
    evaluation.write_sheet_states(states_path, states)
print(f'evaluation states: {states.x.shape[0]}')

In [ ]:
records = self_play.evaluate_checkpoints(
    checkpoint_dir=artifact_dir,
    checkpoint_indices=checkpoint_indices,
    num_states=num_states,
    seed=seed,
    grid_size=grid_size,
    state_generator_version=state_generator_version,
    max_stones=max_stones,
    states=states,
)
records

In [ ]:
import json
results_path = evaluation_dir / 'checkpoint_scores.json'
results_path.write_text(json.dumps(records, indent=2))
for record in records:
    print(
        f"m_{record['checkpoint']:>2} {record['group']:<16} "
        f"mean={record['average_score']:.4f} +/- "
        f"{record['standard_error']:.4f}"
    )